# NeuroObfuscator v6 — Inference & Evaluation (L4 GPU, GGUF)

**Модель**: `gguf_v6/*.gguf` (Qwen2.5-Coder-7B, q4_k_m, LoRA вшиты при экспорте) через **llama.cpp**.
LoRA/merged-safetensors не требуются — оцениваем ровно тот артефакт, который будет задеплоен.

Этапы:
1. Probe формата промпта + sanity generation
2. Quick eval (10 записей)
3. Full eval (850, последовательно, ~30-60 мин) — JSON parse rate, schema rate, latency
4. **Semantic pass rate** — apply + differential validation через Node-движок (нужен `engine_bundle_v6.zip` на Drive)
5. Отчёт JSON + markdown на Drive

Бандл движка: `E:\NEUROOBFUSCATOR\engine_bundle_v6.zip` → загрузить в `MyDrive/neuroobfuscator/`.

Внимание: llama-cpp-python ставится сборкой из исходников с CUDA (в Colab Python 3.13 — готовых wheels нет), первая ячейка занимает ~10 минут.

In [ ]:
import os
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'   # CUDA-сборка llama.cpp (Colab: py3.13 -> только из исходников)
get_ipython().run_line_magic('pip', 'install -q llama-cpp-python')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
ROOT = '/content/drive/MyDrive/neuroobfuscator'
DATA_DIR   = f'{ROOT}/final_v6'
ENGINE_ZIP = f'{ROOT}/engine_bundle_v6.zip'
OUT_DIR    = f'{ROOT}/eval_results_v6'
os.makedirs(OUT_DIR, exist_ok=True)

# GGUF-модель: берём самый крупный .gguf (основной файл, не шард)
ggufs = glob.glob(f'{ROOT}/gguf_v6/*.gguf') or glob.glob(f'{ROOT}/**/*.gguf', recursive=True)
assert ggufs, f'GGUF не найден в {ROOT} — проверь, что gguf_v6 залит на Drive'
GGUF_PATH = max(ggufs, key=os.path.getsize)
print('gguf:', GGUF_PATH, '|', round(os.path.getsize(GGUF_PATH)/1e9, 2), 'GB')
print('data:', sorted(os.listdir(DATA_DIR)))

In [ ]:
import time
from llama_cpp import Llama

t0 = time.time()
llm = Llama(
    model_path=GGUF_PATH,
    n_gpu_layers=-1,      # все слои на GPU (L4 24GB: ~5 GB весов + KV-cache)
    n_ctx=4096,           # промпт <=2048 + генерация 256
    verbose=False,
)
print(f'model loaded in {time.time()-t0:.0f}s')

In [ ]:
import json, re, random, time

SYSTEM_PROMPT = 'You are NeuroObfuscator. Given JavaScript code and its AST features, generate an optimal obfuscation plan as a JSON object.\n\nAvailable transformations (apply in this order when enabled):\n1. rename         - Rename local identifiers to hex-like names. Almost always recommended.\n2. string_encode  - Encode string literals. Methods: charcode_array, charcode_concat, hex_escape, unicode_escape. Only enable if string_count > 0.\n3. operator_sub   - Substitute arithmetic/comparison operators (a+b -> a-(-b), a===b -> !(a!==b)). Use when operator_count > 2.\n4. dead_code      - Insert unreachable code blocks. count: 1-5. More complex code tolerates more.\n5. opaque_predicates - Insert always-true/always-false conditions. count: 1-3. Primarily for medium/heavy intensity; may also be used sparingly on light functions when extra diversity is needed.\n\nIntensity guide:\n- light:  cyclomatic_complexity <= 2. Prefer rename + dead_code only.\n- medium: complexity 3-5. Add string_encode and operator_sub if applicable.\n- heavy:  complexity > 5. Use all relevant transforms aggressively.\n\nRules:\n- Only include enabled transforms in "order" array.\n- Order MUST follow: rename, string_encode, operator_sub, dead_code, opaque_predicates.\n- Do NOT include a "seed" field in your JSON; the runtime injects the provided seed automatically.\n- Avoid over-bloating small functions.\n\nOutput ONLY valid JSON. No explanations, no markdown.'

ORDER = ['rename', 'string_encode', 'operator_sub', 'dead_code', 'opaque_predicates']

def _complexity_class(cc):
    if cc <= 2: return 'light'
    elif cc <= 5: return 'medium'
    return 'heavy'

# --- Форматы промпта. Датасет v6 хранит [INST]-текст; для eval используем instruction
# как есть из jsonl. Свободный код: PROMPT_STYLE выбирает probe-ячейка ниже.
def format_prompt_inst(code, features, seed):
    features_json = json.dumps(features, separators=(',', ':'))
    return (
        f"[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n"
        f"=== CODE ===\n{code}\n=== END CODE ===\n\n"
        f"=== AST FEATURES ===\n{features_json}\n=== END AST FEATURES ===\n\n"
        f"complexity_class={_complexity_class(features.get('cyclomatic_complexity', 1))} "
        f"(cyclomatic_complexity={features.get('cyclomatic_complexity', 1)})\n"
        f"seed={seed}\n\n"
        f"Generate the obfuscation plan JSON: [/INST]"
    )

def format_prompt_chatml(code, features, seed):
    # ChatML Qwen2.5, вручную (tokenizer'а в llama.cpp-стеке нет)
    features_json = json.dumps(features, separators=(',', ':'))
    user = (
        f"=== CODE ===\n{code}\n=== END CODE ===\n\n"
        f"=== AST FEATURES ===\n{features_json}\n=== END AST FEATURES ===\n\n"
        f"complexity_class={_complexity_class(features.get('cyclomatic_complexity', 1))} "
        f"(cyclomatic_complexity={features.get('cyclomatic_complexity', 1)})\n"
        f"seed={seed}\n\nGenerate the obfuscation plan JSON:"
    )
    return (f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{user}<|im_end|>\n"
            f"<|im_start|>assistant\n")

PROMPT_STYLE = 'inst'  # переопределяется probe-ячейкой

def format_prompt(code, features, seed):
    return format_prompt_chatml(code, features, seed) if PROMPT_STYLE == 'chatml' else format_prompt_inst(code, features, seed)

def extract_json(text):
    text = text.strip()
    if text.startswith('{'):
        try: return json.loads(text)
        except json.JSONDecodeError: pass
    decoder = json.JSONDecoder()
    for start, char in enumerate(text):
        if char != '{': continue
        try:
            obj, _ = decoder.raw_decode(text[start:])
            if 'transforms' in obj or 'order' in obj: return obj
        except json.JSONDecodeError: continue
    return None

def validate_plan_schema(plan):
    if not isinstance(plan, dict) or isinstance(plan, list): return False
    if not all(k in plan for k in ['intensity', 'transforms', 'order']): return False
    if plan['intensity'] not in {'light', 'medium', 'heavy'}: return False
    if not isinstance(plan['transforms'], dict) or not isinstance(plan['order'], list): return False
    enabled = [n for n in ORDER if plan['transforms'].get(n, {}).get('enabled')]
    return plan['order'] == enabled

def parse_instruction(instruction):
    """-> (code, features, seed) из текста instruction."""
    code = instruction.split('=== CODE ===\n', 1)[1].split('\n=== END CODE ===', 1)[0]
    feat_json = instruction.split('=== AST FEATURES ===\n', 1)[1].split('\n=== END AST FEATURES ===', 1)[0]
    features = json.loads(feat_json)
    m = re.search(r'seed=(\d+)', instruction)
    seed = int(m.group(1)) if m else random.randint(0, 0xFFFFFFFF)
    return code, features, seed

def generate_one(prompt, max_new_tokens=256):
    """Greedy-генерация (temperature=0)."""
    out = llm(prompt, max_tokens=max_new_tokens, temperature=0.0,
              stop=['<|im_end|>', '</s>'], echo=False)
    return out['choices'][0]['text']

def infer_plan(code, features, seed, retries=2):
    """Модель -> план. Seed НЕ приходит в JSON от модели; инжектируется из промпта."""
    prompt = format_prompt(code, features, seed)
    last_raw = ''
    for _ in range(retries + 1):
        raw = generate_one(prompt)
        last_raw = raw
        plan = extract_json(raw)
        if plan is not None and validate_plan_schema(plan):
            plan['seed'] = seed          # рантайм-инжекция seed для движка
            return plan, raw
    return None, last_raw

print('helpers ready')

In [ ]:
# --- Probe формата промпта (важно только для свободного кода; eval идёт по instruction as-is) ---
import json as _json
with open(f'{DATA_DIR}/test.jsonl', encoding='utf-8') as f:
    records = [_json.loads(l) for l in f if l.strip()]
print('test records:', len(records))

code0, feat0, seed0 = parse_instruction(records[0]['instruction'])
probe = {}
for style, fmt in (('inst', format_prompt_inst), ('chatml', format_prompt_chatml)):
    raw = generate_one(fmt(code0, feat0, seed0))
    plan = extract_json(raw)
    probe[style] = bool(plan and validate_plan_schema(plan))
    print(f'{style}: json+schema ok = {probe[style]} | raw[:80] = {raw[:80]!r}')

PROMPT_STYLE = 'chatml' if probe['chatml'] and not probe['inst'] else 'inst'
print('PROMPT_STYLE =', PROMPT_STYLE)

In [ ]:
# --- Quick eval: 10 записей последовательно ---
from tqdm.notebook import tqdm

stats = {'total': 0, 'ok': 0, 'latencies': []}
failures = []
for rec in tqdm(records[:10], desc='quick eval'):
    code_val, feat_val, seed_val = parse_instruction(rec['instruction'])
    t0 = time.time()
    plan, raw = infer_plan(code_val, feat_val, seed_val)
    stats['total'] += 1
    stats['latencies'].append(time.time() - t0)
    if plan is not None:
        stats['ok'] += 1
    else:
        failures.append(raw[:120])

print(f"JSON+schema: {stats['ok']}/{stats['total']} "
      f"({stats['ok']/max(stats['total'],1):.0%}) | "
      f"avg {sum(stats['latencies'])/len(stats['latencies']):.1f}s/sample")
for s in failures[:3]: print('FAIL:', s)

In [ ]:
# --- Full eval: все записи. llama.cpp не батчит -> последовательно (L4 ~2-4 s/запись) ---
from collections import Counter

t0 = time.time()
raws = [generate_one(r['instruction']) for r in tqdm(records, desc='generate')]
wall = time.time() - t0

plans = []
for rec, raw in zip(records, raws):
    plan = extract_json(raw)
    if plan is not None and validate_plan_schema(plan):
        _, _, seed_val = parse_instruction(rec['instruction'])
        plan['seed'] = seed_val
        plans.append(plan)
    else:
        plans.append(None)

total = len(records)
json_ok = sum(p is not None for p in plans)
intensities = [p['intensity'] for p in plans if p]
orders = [tuple(p['order']) for p in plans if p]

full_eval = {
    'total': total,
    'json_schema_ok': json_ok,
    'invalid': total - json_ok,
    'intensity_dist': dict(Counter(intensities)),
    'unique_orders': len(set(orders)),
    'top_order_share': max(Counter(orders).values()) / max(json_ok, 1),
    'wall_seconds': round(wall, 1),
}
print(f"JSON parse + schema: {json_ok}/{total} = {json_ok/total:.1%} (target >= 95%)")
print('intensity:', full_eval['intensity_dist'])
print('unique orders:', full_eval['unique_orders'],
      '| top-order share:', round(full_eval['top_order_share'], 3))
print(f'wall: {wall/60:.1f} min')

In [ ]:
# --- Сохранить eval-отчёт (промежуточный) ---
import json as _json
_json.dump(full_eval, open(f'{OUT_DIR}/eval_v6_report.json', 'w'), indent=2)
print('saved:', f'{OUT_DIR}/eval_v6_report.json')

In [ ]:
# --- Semantic pass rate: apply + differential validation через Node-движок ---
# Требует engine_bundle_v6.zip на Drive (engine/ + config/ + node_modules/ + package.json)
import shutil, subprocess, os
if shutil.which('node') is None:
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nodejs'], check=False,
                   capture_output=True)
print('node:', subprocess.run(['node', '--version'], capture_output=True, text=True).stdout.strip())

if not os.path.exists('/content/nb/engine/index.js'):
    subprocess.run(['unzip', '-o', '-q', ENGINE_ZIP, '-d', '/content/nb'], check=True)
print(sorted(os.listdir('/content/nb')))

In [ ]:
import subprocess

NB_DIR = '/content/nb'

def call_engine(requests, batch_size=100, timeout=600):
    """Пакетные запросы к engine/index.js --json (JSONL stdin -> JSONL stdout)."""
    results = []
    for s in range(0, len(requests), batch_size):
        chunk = requests[s:s + batch_size]
        payload = '\n'.join(json.dumps(r) for r in chunk)
        proc = subprocess.run(
            ['node', 'engine/index.js', '--json'],
            input=payload, capture_output=True, text=True,
            cwd=NB_DIR, timeout=timeout)
        if proc.returncode != 0:
            print('engine stderr:', proc.stderr[:500])
        results.extend(json.loads(l) for l in proc.stdout.splitlines() if l.strip())
    return results

# sanity: extract_features на примере
sanity = call_engine([{'operation': 'extract_features', 'code': 'function f(a){return a+1;}'}])
print('engine sanity:', sanity[0]['ok'], sanity[0]['value']['cyclomatic_complexity'])

In [ ]:
# --- Прогон семантики на schema-valid планах ---
valid = [(rec, p) for rec, p in zip(records, plans) if p is not None]
print('schema-valid планов:', len(valid), 'из', len(records))

apply_reqs = []
for rec, plan in valid:
    code_val, _, _ = parse_instruction(rec['instruction'])
    apply_reqs.append({'operation': 'apply', 'code': code_val, 'plan': plan})

apply_resp = call_engine(apply_reqs)
obfuscated = [r['value']['code'] if r.get('ok') else None for r in apply_resp]

val_reqs = []
for (rec, _), obf in zip(valid, obfuscated):
    code_val, _, _ = parse_instruction(rec['instruction'])
    if obf is not None:
        val_reqs.append({'operation': 'validate', 'original_code': code_val, 'obfuscated_code': obf})
    else:
        val_reqs.append({'operation': 'extract_features', 'code': 'function x(){}'})  # dummy

val_resp = call_engine(val_reqs)

from collections import Counter
sem = {'total_valid': len(valid), 'apply_failed': 0, 'tests_passed': 0,
       'per_intensity': Counter(), 'per_intensity_total': Counter(), 'reasons': Counter()}
for (rec, _), obf, vresp in zip(valid, obfuscated, val_resp):
    intensity = rec.get('metadata', {}).get('intensity', 'unknown')
    sem['per_intensity_total'][intensity] += 1
    if obf is None:
        sem['apply_failed'] += 1
        continue
    if vresp.get('ok') and vresp['value'].get('tests_passed'):
        sem['tests_passed'] += 1
        sem['per_intensity'][intensity] += 1
    else:
        reason = vresp['value'].get('reason', 'unknown') if vresp.get('ok') else 'engine_error'
        sem['reasons'][reason] += 1

n_valid = sem['total_valid']
sem['apply_ok_rate'] = (n_valid - sem['apply_failed']) / max(n_valid, 1)
sem['semantic_pass_rate'] = sem['tests_passed'] / max(n_valid, 1)
print(f"apply ok:      {n_valid - sem['apply_failed']}/{n_valid} = {sem['apply_ok_rate']:.1%}")
print(f"semantic pass: {sem['tests_passed']}/{n_valid} = {sem['semantic_pass_rate']:.1%}")
print('reasons:', dict(sem['reasons']))
for k in ('light', 'medium', 'heavy'):
    t = sem['per_intensity_total'].get(k, 0)
    if t:
        print(f'  {k}: {sem["per_intensity"][k]}/{t} = {sem["per_intensity"][k]/t:.1%}')

In [ ]:
# --- Итоговый отчёт: JSON + markdown на Drive ---
report = {
    'model': f'Qwen2.5-Coder-7B q4_k_m GGUF ({GGUF_PATH.rsplit(chr(47), 1)[-1]})',
    'dataset': 'final_v6/test.jsonl',
    'prompt_style': PROMPT_STYLE,
    'n_test': full_eval['total'],
    'json_parse_rate': full_eval['json_schema_ok'] / full_eval['total'],
    'schema_rate': full_eval['json_schema_ok'] / full_eval['total'],
    'semantic_pass_rate': sem['semantic_pass_rate'],
    'apply_ok_rate': sem['apply_ok_rate'],
    'intensity_dist': full_eval['intensity_dist'],
    'unique_orders': full_eval['unique_orders'],
    'top_order_share': full_eval['top_order_share'],
    'semantic_by_intensity': {k: (sem['per_intensity'][k], sem['per_intensity_total'][k])
                              for k in ('light', 'medium', 'heavy')},
    'semantic_reasons': dict(sem['reasons']),
    'wall_seconds': full_eval['wall_seconds'],
}
json.dump(report, open(f'{OUT_DIR}/eval_v6_report.json', 'w'), indent=2)

md_lines = [
    '# NeuroObfuscator v6 — Evaluation Report', '',
    f"- Model: {report['model']}",
    f"- Test set: {report['n_test']} records",
    f"- JSON parse rate: **{report['json_parse_rate']:.1%}** (target >= 95%)",
    f"- Schema valid rate: **{report['schema_rate']:.1%}** (target >= 90%)",
    f"- Semantic pass rate: **{report['semantic_pass_rate']:.1%}",
    f"- Intensity dist: {report['intensity_dist']}",
    f"- Unique orders: {report['unique_orders']} | top-order share: {report['top_order_share']:.1%}",
    '', '## Semantic pass by intensity', '',
    '| intensity | passed | total | rate |', '|---|---|---|---|',
]
for k in ('light', 'medium', 'heavy'):
    p, t = report['semantic_by_intensity'][k]
    md_lines.append(f'| {k} | {p} | {t} | {p/max(t,1):.1%} |')
open(f'{OUT_DIR}/eval_v6_report.md', 'w').write('\n'.join(md_lines) + '\n')

print('\n'.join(md_lines))
print('\nsaved:', OUT_DIR)

In [ ]:
# --- Demo: свободный JS-код -> план модели -> обфускация -> validation ---
MY_CODE = """
function calculateDiscount(price, tier) {
  let discount = 0;
  if (tier === 'gold') {
    discount = price * 0.2;
  } else if (tier === 'silver') {
    discount = price * 0.1;
  } else {
    discount = price * 0.05;
  }
  const total = price - discount;
  return Math.round(total * 100) / 100;
}
"""

feat_resp = call_engine([{'operation': 'extract_features', 'code': MY_CODE}])
assert feat_resp[0]['ok'], feat_resp[0].get('error')
features = feat_resp[0]['value']

seed = random.randint(0, 0xFFFFFFFF)
plan, raw = infer_plan(MY_CODE, features, seed)
print('PROMPT_STYLE:', PROMPT_STYLE, '| seed:', seed)
print('PLAN:', json.dumps(plan, indent=2) if plan else f'(invalid) raw: {raw[:200]}')

if plan:
    obf = call_engine([{'operation': 'apply', 'code': MY_CODE, 'plan': plan}])[0]
    if obf.get('ok'):
        val = call_engine([{'operation': 'validate',
                            'original_code': MY_CODE,
                            'obfuscated_code': obf['value']['code']}])[0]
        print('\nVALIDATION:', json.dumps(val['value'], indent=2))
        print('\n--- OBFUSCATED ---')
        print(obf['value']['code'])
    else:
        print('apply failed:', obf.get('error'))